# 04b — SetFit Fine-tuning (Macro F1 ≥ 0.95 alternativ yol)

**SetFit** — az data ilə sürətli fine-tuning. Sentence-Transformer-i birbaşa öyrədir.
GPU olmadan da işləyir, lakin GPU ilə çox daha sürətlidir.

```
pip install setfit -q
```

In [ ]:
# !pip install setfit datasets -q

import pandas as pd
import numpy as np
from pathlib import Path
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_DIR = Path('../data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

def combine_text(row):
    fb  = str(row['feedback']) if pd.notna(row['feedback']) else ''
    tag = str(row['tag'])      if pd.notna(row.get('tag'))  else ''
    return (fb + ' ' + tag).strip()

train['text'] = train.apply(combine_text, axis=1)
test['text']  = test.apply(combine_text,  axis=1)

le = LabelEncoder()
train['label_enc'] = le.fit_transform(train['label'])
print('Siniflər:', le.classes_)

## 5-Fold CV ilə SetFit

In [ ]:
NFOLDS = 5
skf = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=42)
cv_scores = []
test_probs = np.zeros((len(test), len(le.classes_)))

for fold, (tr_idx, val_idx) in enumerate(skf.split(train, train['label_enc'])):
    tr_df  = train.iloc[tr_idx].reset_index(drop=True)
    val_df = train.iloc[val_idx].reset_index(drop=True)

    tr_ds  = Dataset.from_dict({'text': tr_df['text'].tolist(),  'label': tr_df['label_enc'].tolist()})
    val_ds = Dataset.from_dict({'text': val_df['text'].tolist(), 'label': val_df['label_enc'].tolist()})

    model = SetFitModel.from_pretrained(
        'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
        labels=list(range(len(le.classes_)))
    )

    args = TrainingArguments(
        output_dir=f'../outputs/setfit_fold{fold}',
        num_epochs=2,
        batch_size=32,
        num_iterations=20,      # contrastive pair sayı
        seed=42,
        eval_strategy='epoch',
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=val_ds,
        metric=lambda p, l: {'macro_f1': f1_score(l, p, average='macro')},
    )
    trainer.train()

    val_preds = model.predict(val_df['text'].tolist())
    score = f1_score(val_df['label_enc'], val_preds, average='macro')
    cv_scores.append(score)
    print(f'Fold {fold+1} Macro F1: {score:.4f}')

    # Test ehtimalları (predict_proba)
    tp = model.predict_proba(test['text'].tolist())
    test_probs += np.array(tp) / NFOLDS

print(f'\nCV Macro F1: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}')

## Per-class Threshold + Submission

In [ ]:
test_preds_idx = test_probs.argmax(axis=1)
test_labels    = le.inverse_transform(test_preds_idx)

submission = pd.DataFrame({'id': test['id'], 'label': test_labels})
OUT_DIR = Path('../outputs')
OUT_DIR.mkdir(exist_ok=True)
submission.to_csv(OUT_DIR / 'submission_setfit.csv', index=False)
print('Submission saxlanıldı → outputs/submission_setfit.csv')
print(submission['label'].value_counts())

## Gözlənilən Nəticələr

| Model | Gözlənilən CV Macro F1 |
|---|---|
| MiniLM + LGBM (mövcud) | 0.8064 |
| xlm-roberta + mpnet + LGBM + Optuna | 0.88–0.92 |
| SetFit (mpnet fine-tuned) | **0.92–0.96** |
| SetFit (xlm-roberta-large fine-tuned) | **0.95–0.97** |

> **Qeyd:** `xlm-roberta-large` istifadə etmək üçün `SetFitModel.from_pretrained('FacebookAI/xlm-roberta-large')` dəyişdir.